# PUBG API Data Extraction & Structuring (Python Version)
This notebook tests the extraction of PUBG match data and structures it into flattened Pandas DataFrames. It includes high-resolution telemetry data.

In [4]:
import requests
import pandas as pd
import json
import time
from config import api_key
from IPython.display import display

In [5]:
valid_shards = ["steam", "console"]

### Steam Data Extraction

In [6]:
##Fetches a list of recent match IDs from the PUBG API (It is the only way to get a random, high-volume list of matches without knowing specific players beforehand)

def get_pubg_samples_steam(api_key, shard=valid_shards[0]):
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [7]:
match_samples = get_pubg_samples_steam(api_key)

print(match_samples)


{'data': {'type': 'sample', 'id': 'ed0e64c0-4f47-434f-9e74-e98292b5ea0c', 'attributes': {'createdAt': '2026-05-02T00:00:00Z', 'titleId': 'bluehole-pubg', 'shardId': 'steam'}, 'relationships': {'matches': {'data': [{'type': 'match', 'id': 'a6f500a1-77e0-43d6-9421-5c886be34f73'}, {'type': 'match', 'id': '73508639-0151-480f-9493-d0b7d6333fb6'}, {'type': 'match', 'id': '911ad460-b690-4a08-9b85-ac30bc60e437'}, {'type': 'match', 'id': 'a2f1c175-afcc-427e-aec9-e54bd8da1312'}, {'type': 'match', 'id': '7952d69e-843a-4d81-b96d-2d09d1cbaa01'}, {'type': 'match', 'id': 'ed594925-28e3-408e-b38d-64d067637eb9'}, {'type': 'match', 'id': '8eb6de1f-ed93-47f9-9be6-2a0aa006d4fd'}, {'type': 'match', 'id': 'dff82313-cc7d-4ac4-ab19-b205414cf2b1'}, {'type': 'match', 'id': '5e37bba7-0ca2-47e4-b2e7-45a9c00126a6'}, {'type': 'match', 'id': '53f827a6-7c95-4b74-85e4-1ec2eca7e840'}, {'type': 'match', 'id': '9fc90407-7992-4ff6-b2f5-01ca64a182f3'}, {'type': 'match', 'id': 'b0d6abcb-adb4-4181-831d-d67c4adf175b'}, {'type

In [ ]:
matches_list = match_samples['data']['relationships']['matches']['data']
matches_list.rename(columns={'id': 'match_id'}, inplace=True)
match_samples_df = pd.DataFrame(matches_list)
print(f"Total matches retrieved: {len(match_samples_df)}")
display(match_samples_df.head())

Total matches retrieved: 800


,type,id
0,match,a6f500a1-77e0-43d6-9421-5c886be34f73
1,match,73508639-0151-480f-9493-d0b7d6333fb6
2,match,911ad460-b690-4a08-9b85-ac30bc60e437
3,match,a2f1c175-afcc-427e-aec9-e54bd8da1312
4,match,7952d69e-843a-4d81-b96d-2d09d1cbaa01


In [10]:
match_id = match_samples_df['id'].tolist()

In [11]:
def get_match_details(api_key, match_id, shard=valid_shards[0]):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [25]:
match_details = get_match_details(api_key,match_id[0])
print(match_details)

{'data': {'type': 'match', 'id': '95ff792e-3728-49c8-9ca2-6e1364af7639', 'attributes': {'shardId': 'steam', 'tags': None, 'isCustomMatch': False, 'createdAt': '2026-04-29T23:42:44Z', 'stats': None, 'titleId': 'bluehole-pubg', 'mapName': 'Tiger_Main', 'matchType': 'official', 'seasonState': 'progress', 'duration': 1532, 'gameMode': 'duo-fpp'}, 'relationships': {'assets': {'data': [{'type': 'asset', 'id': 'f551103a-4428-11f1-ba3a-667ef378d19f'}]}, 'rosters': {'data': [{'type': 'roster', 'id': 'f7cb0d8f-280b-49a2-9339-8a8c1f915e2f'}, {'type': 'roster', 'id': '3e38ea4b-d93e-4cc4-9366-dd38f7f0f7c5'}, {'type': 'roster', 'id': '1d774de1-26cf-4b12-9238-3829a4e0517f'}, {'type': 'roster', 'id': 'c870f0a6-073e-4cef-b64c-9ef917c99e46'}, {'type': 'roster', 'id': '983fcbf5-b6c8-4db7-8b6a-52f5a1e4e06b'}, {'type': 'roster', 'id': '208ce643-c1b5-4694-9de1-8f429d09c131'}, {'type': 'roster', 'id': 'c358ccd3-4ca6-46d1-bfef-4e216c6df11f'}, {'type': 'roster', 'id': '68f77f4f-03f2-4411-b96b-a1fea896c902'}, {

In [21]:
match_details = get_match_details(api_key,match_id[0])
matches_details_list = match_details['data']['relationships']['assets']['data']
print(matches_details_list)

[{'type': 'asset', 'id': 'f551103a-4428-11f1-ba3a-667ef378d19f'}]


In [39]:
def extract_telemetry_url(match_details):
    included_items = match_details.get("included", [])
    
    for item in included_items:
        is_telemetry = item.get("attributes", {}).get("name") == "telemetry"
        
        if is_telemetry:
            telemetry_url = item["attributes"]["URL"]
            print("Telemtry URL sucessfully extracted!")
            return telemetry_url
            
    print("Did not find telemetry URL in match details:(")
    return None

In [ ]:
telemetry_url = extract_telemetry_url(match_details)

Telemtry URL sucessfully extracted!


In [41]:
telemetry_url_df = pd.DataFrame(telemetry_url)
display(telemetry_url_df.head())

ValueError: DataFrame constructor not properly called!